# Sports market calibration — directly from the CSV

This notebook reads the original wide CSV each time it runs. It uses hours **2–24 before close**, excluding the final hour. A market is counted **once in each 10% bin it reaches**, even when several hours fall in that bin; it may be counted in multiple bins. Its probability in a bin is the mean of its hourly prices in that bin. Only settled binary markets with usable hourly prices are included.

Run from the repository root. The CSV is large, so the first computation can take time; progress is printed as it reads the file. No pickle is read or saved.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from plot_event_sport_calibration import hourly_market_bins, calibration_bins

CSV = Path('Data/all_closed_sports/closed_sports_all_data.csv')
if not CSV.is_file():
    raise FileNotFoundError(f'Cannot find the local CSV: {CSV.resolve()}')

## Read the hours and count market presence

The CSV stays in its original one-row-per-market form. `hourly_market_bins` reads only the necessary columns in chunks, converts hourly trade VWAPs to first-outcome probabilities, and temporarily groups them by market and bin. If only the second outcome traded, its probability is converted using `1 − price`. The resulting `market_bins` is held in memory only. The helper is in `plot_event_sport_calibration.py` in this repository.

In [ ]:
market_bins = hourly_market_bins(CSV, n_bins=10, start_hour=2)
if market_bins.empty:
    raise ValueError('No usable settled market-hour observations were found.')
if market_bins.duplicated(['market_id', 'probability_bin']).any():
    raise ValueError('A market was counted more than once in the same bin.')
print(f"{market_bins.market_id.nunique():,} unique markets; {len(market_bins):,} market-bin presences")

bins = calibration_bins(market_bins, n_bins=10).reset_index()
bins['range'] = bins['probability_bin'].map(lambda b: f'{10*b}–{10*(b+1)}%')
bins[['range', 'n', 'mean_probability', 'observed_rate', 'ci_low', 'ci_high']]

## Calibration plot

The point is the observed win frequency versus the average predicted probability within a bin. The count is the number of distinct markets present in that bin. Wilson 95% intervals assume independent observations; markets on the same event can be related, so these intervals can understate uncertainty. The weighted regression is descriptive and fitted to these same bin averages.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))
ax.plot([0, 1], [0, 1], '--', color='0.45', label='Perfect calibration')
if len(bins) >= 2:
    slope, intercept = np.polyfit(bins.mean_probability, bins.observed_rate, 1, w=np.sqrt(bins.n))
    grid = np.linspace(0, 1, 200)
    ax.plot(grid, intercept + slope * grid, color='#c06b32', label=f'Weighted regression (slope = {slope:.2f})')
yerr = np.maximum(0, np.array([bins.observed_rate - bins.ci_low, bins.ci_high - bins.observed_rate]))
ax.errorbar(bins.mean_probability, bins.observed_rate, yerr=yerr, fmt='o',
            color='#3569a8', markersize=5, capsize=4, label='10% bins; Wilson 95% CI')
for row in bins.itertuples():
    ax.annotate(f'n={row.n:,}', (row.mean_probability, row.observed_rate),
                xytext=(5, 5), textcoords='offset points', fontsize=8)
ax.set(xlim=(0, 1), ylim=(0, 1), xlabel='Predicted probability',
       ylabel='Observed event frequency',
       title=f'Sports market calibration ({market_bins.market_id.nunique():,} markets)')
ax.grid(alpha=.25)
ax.legend()
fig.tight_layout()
plt.show()

## Number of markets present in each bin

A market may occur in more than one bar, so the bar counts do not add up to the number of unique markets.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(bins['range'], bins['n'], color='#3569a8')
ax.set(xlabel='Probability bin', ylabel='Distinct markets', title='Market presence by probability bin')
ax.grid(axis='y', alpha=.25)
fig.tight_layout()
plt.show()